# ATML PA1 -- Colab bootstrap

Run this once per session. It mounts Drive (for datasets/checkpoints, which are git-ignored
and must survive a disconnect), clones/pulls this repo (the source of truth for code and
directory structure), and installs dependencies. After this, run everything as
`!python -m task2.train --config task2/configs/dann.yaml`-style commands from the repo root
so the exact same scripts work locally too.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/atml_pa1'
DATA_ROOT = f'{DRIVE_ROOT}/data'       # STL-10 / PACS / CIFAR-10 / CIFAR-100 cache
CKPT_ROOT = f'{DRIVE_ROOT}/checkpoints' # everything task*/results/*/checkpoint.pt points at

import os
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)
print('Data root:', DATA_ROOT)
print('Checkpoint root:', CKPT_ROOT)

In [ ]:
REPO_URL = 'https://github.com/adilawan1/ATML-PA1.git'
REPO_DIR = '/content/ATML-PA1'

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git pull
else:
    !git clone $REPO_URL $REPO_DIR
    %cd $REPO_DIR

## Git identity + GitHub auth (run once per session, before any commit cell)

Colab starts with neither a git identity nor GitHub push credentials, and neither survives a
runtime reset -- this needs to run every fresh session, not just once ever.

1. If you don't already have one, create a GitHub Personal Access Token (classic, `repo`
   scope) at github.com/settings/tokens.
2. In this notebook's left sidebar, click the key icon ("Secrets"), add a new secret named
   `GH_TOKEN` with that token as the value, and enable notebook access for it.
3. Run the cell below (edit the placeholder name on its first line first).

In [ ]:
# EDIT the name below to your real name first -- your local machine's git config has
# user.name set to a stray "=", don't copy that here.
!git config --global user.name "YOUR NAME HERE"
!git config --global user.email "1ahmed2adil3awan@gmail.com"

from google.colab import userdata
GH_TOKEN = userdata.get('GH_TOKEN')
!git remote set-url origin https://{GH_TOKEN}@github.com/adilawan1/ATML-PA1.git
print("git identity + auth configured for this session")

In [ ]:
# IMPORTANT: do NOT `pip install torch`/`torchvision` here. Colab preinstalls a build of
# each already matched to its GPU driver; PyPI's default (untagged) wheel for both is
# CPU-only, and reinstalling them is exactly what causes
# "AssertionError: Torch not compiled with CUDA enabled" later on. Install everything else
# from requirements.txt and leave those two alone.
!grep -vE '^(torch|torchvision)$' requirements.txt > /tmp/requirements_colab.txt
!pip install -q -r /tmp/requirements_colab.txt

# Sanity check -- if this ever prints False/None on a GPU runtime, something (re)installed a
# CPU-only torch; fix with:
#   !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
import torch
print("torch", torch.__version__, "| cuda build:", torch.version.cuda, "| available:", torch.cuda.is_available())

In [ ]:
!nvidia-smi

## Overnight batch (Tasks 2, 3, PROSER, Task 1 interventions -- unattended)

For a long unattended run use **this cell instead of "Run all"**. It needs the setup cells above (mount Drive, clone, install) but
nothing else, and runs in the order of importance: Task 2 -> Task 3 -> PROSER -> Task 1 interventions (about 3.5-4 h on a T4).
Keep the browser tab open with the machine awake and plugged in -- background execution (running with the tab closed) is, as far
as I know, a Colab Pro+ feature, not Pro; check your plan. What survives a disconnect: Drive (checkpoints, feature caches, the
per-stage result backups). What does not: `/content` (PACS folders, preprocessed caches, the clone and its result files).

In [ ]:
# -------------------------------------------------------------------------------------------------
# Unattended batch. Run the setup cells above first (mount, clone, install), then run ONLY this cell
# -- not "Run all": that would stop at the first guarded commit cell and would generate cue conflicts
# before you have calibrated them. Nothing is committed here (commits come after your hypotheses).
# Every stage is resumable, backed up to Drive when it ends, and a failing stage does not stop the next.
# -------------------------------------------------------------------------------------------------
TASK2_STUDY = "dan"   # "dan" or "dann"    -- design choices (the hypotheses about them come later)
TASK3_STUDY = "sam"   # "sam" or "dan_dg"
# Vanilla/GCSC were first trained before seeding was added, but the spec fixes seed 6304. True retrains them (and
# PROSER, which starts from Vanilla) seeded -- about 2 extra hours -- keeping your old checkpoints as
# checkpoint_unseeded_backup.pt on Drive. Set False to keep the unseeded ones (then say so in the report).
RETRAIN_TASK4_SEEDED = True
B = f"{DRIVE_ROOT}/results_backup"
PACS_PARQUET = f"{DATA_ROOT}/pacs_flwrlabs.parquet"

# --- PACS on local disk; the committed split is used as is (it is only validated against these files) ---
!python -m shared.prepare_pacs --out /content/pacs --parquet {PACS_PARQUET}
!python -m shared.pacs_protocol --root /content/pacs
!mkdir -p {B}

# --- Task 2 (~1 h) ---
!python -m task2.run_experiments --pacs-root /content/pacs --ckpt-root {CKPT_ROOT} --study {TASK2_STUDY} --blind
!rm -rf {B}/task2 {B}/figures && cp -r task2/results {B}/task2 && cp -r report/figures {B}/figures

# --- Task 3 (~45 min; needs Task 2's Source-only checkpoint) ---
!python -m task3.run_experiments --pacs-root /content/pacs --ckpt-root {CKPT_ROOT} --study {TASK3_STUDY} --blind
!python -m task3.evaluate_sketch --pacs-root /content/pacs --ckpt-root {CKPT_ROOT} --blind
!rm -rf {B}/task3 {B}/figures && cp -r task3/results {B}/task3 && cp -r report/figures {B}/figures

# --- Task 4: (seeded) Vanilla + GCSC, PROSER, final tables (~45 min, or ~3 h with the seeded retrain) ---
import glob, traceback
try:
    from task4.evaluate_osr import main as evaluate_osr_main
    from task4.extract_outputs import extract_all_outputs
    from task4.methods.gcsc import train_gcsc
    from task4.methods.proser import train_proser
    from task4.methods.vanilla import train_vanilla

    SPLIT = "task4/data/cifar10_split_seed6304.json"
    CKPTS = {n: f"{CKPT_ROOT}/task4/{n}/checkpoint.pt" for n in ("vanilla", "gcsc", "proser")}
    TRAIN = {
        "vanilla": lambda: train_vanilla(data_root=DATA_ROOT, split_path=SPLIT, device="cuda", checkpoint_path=CKPTS["vanilla"], metrics_path="task4/results/vanilla/metrics.jsonl"),
        "gcsc": lambda: train_gcsc(data_root=DATA_ROOT, split_path=SPLIT, device="cuda", checkpoint_path=CKPTS["gcsc"], metrics_path="task4/results/gcsc/metrics.jsonl"),
        "proser": lambda: train_proser(data_root=DATA_ROOT, vanilla_checkpoint=CKPTS["vanilla"], device="cuda", checkpoint_path=CKPTS["proser"], metrics_path="task4/results/proser/metrics.jsonl"),
    }
    for f in glob.glob("task4/cache/*.pt"):   # never reuse features extracted from an older checkpoint
        os.remove(f)
    for name in ("vanilla", "gcsc", "proser"):   # vanilla first: PROSER starts from it
        ckpt, marker = CKPTS[name], CKPTS[name] + ".seeded"
        if os.path.exists(marker):
            continue                                   # already trained with seed 6304
        if RETRAIN_TASK4_SEEDED or not os.path.exists(ckpt):
            backup = ckpt.replace("checkpoint.pt", "checkpoint_unseeded_backup.pt")
            if os.path.exists(ckpt):
                os.remove(ckpt) if os.path.exists(backup) else os.rename(ckpt, backup)
            TRAIN[name]()
            open(marker, "w").write("trained with seed 6304\n")
    for name, ckpt in CKPTS.items():
        if os.path.exists(ckpt):
            extract_all_outputs(ckpt, DATA_ROOT, name, device="cuda")
    evaluate_osr_main(data_root=DATA_ROOT, cache_dir="task4/cache")
except Exception:
    traceback.print_exc()
!rm -rf {B}/task4 {B}/figures && cp -r task4/results {B}/task4 && cp -r report/figures {B}/figures

# --- Task 1 interventions, blind (STL-10 download + features, ~30-40 min). Cue conflicts stay manual. ---
!python -m task1.scripts.run_task1 --data-root {DATA_ROOT} --cache-dir {DRIVE_ROOT}/cache/task1 --blind
!rm -rf {B}/task1 {B}/figures && cp -r task1/results {B}/task1 && cp -r report/figures {B}/figures && cp task1/data/eval_subset_seed6304.json task1/data/stl10_train_val_split_seed6304.json {B}/
print("BATCH FINISHED")

**If the runtime was recycled overnight**, run the next cell first (before the commit cells): it restores the small result
files from the Drive backup into the fresh clone. Skip it if the session survived.

In [ ]:
# Only needed if the runtime was recycled overnight: copy the backed-up small results back into the fresh clone.
# (Checkpoints and feature caches live on Drive and survive; the PACS folders are rebuilt by the batch cell.)
B = f"{DRIVE_ROOT}/results_backup"
for stage in ["task1", "task2", "task3", "task4"]:
    !mkdir -p {stage}/results && cp -r {B}/{stage}/. {stage}/results/
!mkdir -p report/figures && cp -r {B}/figures/. report/figures/
!cp {B}/eval_subset_seed6304.json {B}/stl10_train_val_split_seed6304.json task1/data/ 2>/dev/null; ls task2/results task3/results | head

## Task 4 kickoff (Vanilla + GCSC)

These are the two required Task 4 trainings that need no manual dataset download (CIFAR-10
fetches automatically) and don't depend on PACS. Each cell checks for an existing checkpoint
first, so it's safe to re-run this section if the runtime disconnects mid-way -- it will skip
anything already finished rather than retraining from scratch. Run the three cells below in
order, then the commit cell once both are done.

In [ ]:
import os

SPLIT_PATH = "task4/data/cifar10_split_seed6304.json"
if not os.path.exists(SPLIT_PATH):
    !python -m task4.data.make_splits --data-root {DATA_ROOT}
else:
    print(f"{SPLIT_PATH} already exists, skipping.")

In [ ]:
import os

from task4.methods.vanilla import train_vanilla

VANILLA_CKPT = f"{CKPT_ROOT}/task4/vanilla/checkpoint.pt"
if os.path.exists(VANILLA_CKPT):
    print(f"Found existing checkpoint at {VANILLA_CKPT}, skipping training. Delete it to retrain.")
else:
    vanilla_model = train_vanilla(
        data_root=DATA_ROOT,
        split_path="task4/data/cifar10_split_seed6304.json",
        device="cuda",
        checkpoint_path=VANILLA_CKPT,
        metrics_path="task4/results/vanilla/metrics.jsonl",
    )

In [ ]:
import os

from task4.methods.gcsc import train_gcsc

GCSC_CKPT = f"{CKPT_ROOT}/task4/gcsc/checkpoint.pt"
if os.path.exists(GCSC_CKPT):
    print(f"Found existing checkpoint at {GCSC_CKPT}, skipping training. Delete it to retrain.")
else:
    gcsc_model = train_gcsc(
        data_root=DATA_ROOT,
        split_path="task4/data/cifar10_split_seed6304.json",
        device="cuda",
        checkpoint_path=GCSC_CKPT,
        metrics_path="task4/results/gcsc/metrics.jsonl",
    )

In [ ]:
!git pull
!git add task4/data/cifar10_split_seed6304.json task4/results/vanilla/metrics.jsonl task4/results/gcsc/metrics.jsonl
!git commit -m "Add Task 4 Vanilla and GCSC training curves"
!git push

## Task 4 evaluation (run once Vanilla + GCSC checkpoints exist)

Caches each checkpoint's features/logits, then builds both required tables (post-hoc scores
on Vanilla; Vanilla/GCSC comparison via MLS), the score-distribution figure, and the
near/far failure cases. Safe to re-run -- `extract_outputs` overwrites its own cache files,
and `evaluate_osr` just recomputes tables from whatever caches currently exist (it'll pick up
a PROSER row automatically once that's trained and cached the same way).

In [ ]:
from task4.extract_outputs import extract_all_outputs

extract_all_outputs(
    checkpoint_path=VANILLA_CKPT,
    data_root=DATA_ROOT,
    method_name="vanilla",
    device="cuda",
)
extract_all_outputs(
    checkpoint_path=GCSC_CKPT,
    data_root=DATA_ROOT,
    method_name="gcsc",
    device="cuda",
)

In [ ]:
from task4.evaluate_osr import main as evaluate_osr_main

evaluate_osr_main(data_root=DATA_ROOT, cache_dir="task4/cache")

In [ ]:
!git pull
!git add task4/results/table1_vanilla_posthoc_scores.json task4/results/table2_model_comparison_mls.json task4/results/failure_cases.json report/figures/task4_score_distributions.png
!git commit -m "Add Task 4 Vanilla/GCSC evaluation tables, figure, and failure cases"
!git push

## PACS dataset (needed for Tasks 2 and 3)

Uses the Hugging Face copy of the standard PACS release (`flwrlabs/pacs`, 9,991 images) -- no Drive
quota or Kaggle account involved. The parquet file is cached on Drive; the image folders are built
on fast **local** disk (`/content/pacs`, ~1 min) at the start of every new session, because reading
thousands of small files through Drive during training is slow. Then verify the layout/image counts
(Photo 1670, Art 2048, Cartoon 2344, Sketch 3929) and build/commit the shared split protocol.

In [ ]:
PACS_PARQUET = f"{DATA_ROOT}/pacs_flwrlabs.parquet"   # cached on Drive (~190 MB)
PACS_DIR = "/content/pacs"                            # rebuilt on local disk each new session
!python -m shared.prepare_pacs --out {PACS_DIR} --parquet {PACS_PARQUET}

In [ ]:
from shared.verify_pacs import verify

PACS_DIR = "/content/pacs"   # built by the cell above
assert verify(PACS_DIR), "Fix the PACS layout above before building the split protocol"
!python -m shared.pacs_protocol --root {PACS_DIR}

!git pull
!git add shared/splits/pacs_sketch_seed6304.json
!git commit -m "Add PACS split protocol (seed 6304; Photo/Art/Cartoon source, Sketch target)"
!git push

## Task 1 -- clean baseline, color, translation, patch shuffle, representation analysis

The run is **blind** (no results table printed). The assignment wants each design choice's hypothesis stated before its result
is *interpreted*: write yours in `task1/hypotheses.md` and commit it before you open the results -- the commit cell below
refuses to push results until you have. STL-10 downloads (~2.6 GB) into `DATA_ROOT` the first time; frozen train/val features are
cached on Drive so re-runs skip extraction. Cue conflicts (Step 3) are the next section.

In [ ]:
!python -m task1.scripts.run_task1 --data-root {DATA_ROOT} --cache-dir {DRIVE_ROOT}/cache/task1 --blind
# results stay unread (and uncommitted) until your hypotheses are committed; keep a copy on Drive meanwhile
!rm -rf {DRIVE_ROOT}/results_backup/task1 && mkdir -p {DRIVE_ROOT}/results_backup && cp -r task1/results {DRIVE_ROOT}/results_backup/task1

In [ ]:
!git pull
from shared.preregistration import assert_hypotheses_committed
assert_hypotheses_committed("task1/hypotheses.md")   # results may only be committed AFTER your hypotheses are
!git add task1/data/eval_subset_seed6304.json task1/data/stl10_train_val_split_seed6304.json task1/results/task1_results.json task1/results/compact_comparison.csv report/figures/task1_translation_curve.png report/figures/task1_tsne.png
!git commit -m "Add Task 1 results: clean baseline, color, translation, patch shuffle, representation stability"
!git push

## Task 1, Step 3 -- shape vs. texture cue conflicts (AdaIN)

Order matters -- the assignment forbids using model predictions to decide which conflicts to keep:
1. **Preview** (images + image statistics only, no classifier): pick the style strength `alpha` and
   the rejection-rule thresholds by eye from the contact sheet.
2. Put your choices in `task1/configs/task1.yaml` (`cue_conflicts:`), fill in `task1/hypotheses.md`,
   commit and push from your PC, then `!git pull` here. **From this point the rule is frozen.**
3. **Generate** the accepted set (no classifier is loaded), **evaluate** all four predictors on it,
   then commit the results.

The first run clones the public pytorch-AdaIN repo (pinned commit) into `third_party/` and downloads
its two weight files (~94 MB).

In [ ]:
!python -m task1.data.make_cue_conflicts --data-root {DATA_ROOT} --preview
# Open report/figures/task1_cue_conflict_preview.png (Colab file browser) and judge the images.

In [ ]:
!python -m task1.data.make_cue_conflicts --data-root {DATA_ROOT} --cache-dir {DRIVE_ROOT}/cache/task1

In [ ]:
!python -m task1.scripts.run_cue_conflicts --data-root {DATA_ROOT} --cache-dir {DRIVE_ROOT}/cache/task1 --blind
!rm -rf {DRIVE_ROOT}/results_backup/task1_cue && cp -r task1/results {DRIVE_ROOT}/results_backup/task1_cue

In [ ]:
!git pull
from shared.preregistration import assert_hypotheses_committed
assert_hypotheses_committed("task1/hypotheses.md")   # results may only be committed AFTER your hypotheses are
!git add task1/results/cue_conflicts_meta.json task1/results/cue_conflicts.json task1/results/cue_conflicts_summary.csv report/figures/task1_cue_conflict_accepted.png report/figures/task1_cue_conflict_rejected.png report/figures/task1_cue_conflict_examples.png report/figures/task1_tsne_cue_conflict.png
!git commit -m "Add Task 1 cue-conflict results (AdaIN)"
!git push

## Task 2 -- Source-only, DAN, DANN, CDAN, your chosen controlled study

Needs the PACS section above to have run in this session. Choose `TASK2_STUDY`, then launch: the run is **blind**
(no metrics are printed), so it can start now while you read. The assignment requires your expected effect of stronger
alignment to be stated *before you interpret the results* -- write it in `task2/hypotheses.md` and commit it, and only then
open the results; the commit cell below refuses to push results until you have. Roughly an hour on a T4; resumable, so
re-running after a disconnect skips finished runs. Target labels are first read in the final evaluation inside this
script; never adjust a setting after looking at its output.

In [ ]:
TASK2_STUDY = "dan"   # "dan" (lambda_MMD in {0.1, 1, 10}) or "dann" (max GRL strength in {0.25, 0.5, 1}) -- your choice
# --blind: no metrics are printed, so you can launch now and read the results only after writing your hypotheses
!python -m task2.run_experiments --pacs-root /content/pacs --ckpt-root {CKPT_ROOT} --study {TASK2_STUDY} --blind
!rm -rf {DRIVE_ROOT}/results_backup/task2 && mkdir -p {DRIVE_ROOT}/results_backup && cp -r task2/results {DRIVE_ROOT}/results_backup/task2

In [ ]:
!git pull
from shared.preregistration import assert_hypotheses_committed
assert_hypotheses_committed("task2/hypotheses.md")   # results may only be committed AFTER your hypotheses are
!git add task2/results report/figures/task2_*.png
!git commit -m "Add Task 2 results: Source-only, DAN, DANN, CDAN, controlled study"
!git push

## Task 3 -- ERM, DAN-DG, SAM, your chosen controlled study

Needs the Task 2 Source-only checkpoint on Drive (ERM *is* that checkpoint; it is never retrained here). Runs are **blind**
(no metrics printed); write your expectations in `task3/hypotheses.md` and commit them before you read any result -- do this
before opening Task 2's results too, since Task 2's Sketch numbers must not influence any Task 3 setting. The first cell trains
and computes the source-side diagnostics **without any Sketch image being loaded**; only the second cell (`evaluate_sketch`)
ever opens Sketch.

In [ ]:
TASK3_STUDY = "sam"   # "sam" (rho in {0.01, 0.05, 0.1}) or "dan_dg" (lambda_DG in {0.1, 1, 10}) -- your choice
!python -m task3.run_experiments --pacs-root /content/pacs --ckpt-root {CKPT_ROOT} --study {TASK3_STUDY} --blind

In [ ]:
!python -m task3.evaluate_sketch --pacs-root /content/pacs --ckpt-root {CKPT_ROOT} --blind
!rm -rf {DRIVE_ROOT}/results_backup/task3 && mkdir -p {DRIVE_ROOT}/results_backup && cp -r task3/results {DRIVE_ROOT}/results_backup/task3

In [ ]:
!git pull
from shared.preregistration import assert_hypotheses_committed
assert_hypotheses_committed("task3/hypotheses.md")   # results may only be committed AFTER your hypotheses are
!git add task3/results report/figures/task3_*.png
!git commit -m "Add Task 3 results: ERM, DAN-DG, SAM, controlled study, Sketch evaluation"
!git push

## Task 4 -- PROSER (classifier + data placeholders), then the final tables

Initializes from the selected Vanilla checkpoint, appends 5 dummy classifiers, fine-tunes 50 epochs
(SGD lr 1e-3, cosine, batch 128, seed 6304; beta = 1, gamma = 0.1). Then extracts features for all three
models and regenerates the Task 4 tables, which now include PROSER twice: MLS on its ten known-class logits
and its placeholder-based score. Roughly 30-45 min on a T4.

Note: Vanilla and GCSC were first trained before seeding was moved inside `train_vanilla`, so those two runs
were not seeded even though the spec asks for seed 6304. If you have GPU time to spare, delete the two Vanilla /
GCSC checkpoints on Drive, re-run the "Task 4 kickoff" cells (about 1-2 h), and then delete the PROSER checkpoint
and re-run this section -- everything downstream regenerates. Otherwise say so in the report's limitations.

In [ ]:
import os
from task4.methods.proser import train_proser

VANILLA_CKPT = f"{CKPT_ROOT}/task4/vanilla/checkpoint.pt"
PROSER_CKPT = f"{CKPT_ROOT}/task4/proser/checkpoint.pt"
if os.path.exists(PROSER_CKPT):
    print(f"Found existing checkpoint at {PROSER_CKPT}, skipping training. Delete it to retrain.")
else:
    train_proser(
        data_root=DATA_ROOT,
        vanilla_checkpoint=VANILLA_CKPT,
        device="cuda",
        checkpoint_path=PROSER_CKPT,
        metrics_path="task4/results/proser/metrics.jsonl",
    )

In [ ]:
import os
from task4.evaluate_osr import main as evaluate_osr_main
from task4.extract_outputs import extract_all_outputs

CKPTS = {
    "vanilla": f"{CKPT_ROOT}/task4/vanilla/checkpoint.pt",
    "gcsc": f"{CKPT_ROOT}/task4/gcsc/checkpoint.pt",
    "proser": f"{CKPT_ROOT}/task4/proser/checkpoint.pt",
}
# Feature/logit caches live on the VM disk, so a fresh session re-extracts them (about a minute each).
for name, ckpt in CKPTS.items():
    if os.path.exists(ckpt) and not os.path.exists(f"task4/cache/{name}_far.pt"):
        extract_all_outputs(ckpt, DATA_ROOT, name, device="cuda")
evaluate_osr_main(data_root=DATA_ROOT, cache_dir="task4/cache")

In [ ]:
!git pull
!git add task4/results report/figures/task4_*.png
!git commit -m "Add PROSER; regenerate Task 4 tables with PROSER rows"
!git push

## Final freeze -- record the environment and audit the submission

Run this last, after every results-commit cell. It records the Python / library / GPU versions the results were produced with
(`env/`), refreshes `report/evidence_map.md` (which committed file backs each required item), commits both, and then runs
`tools/check_submission.py`, which reports PASS / MISSING / WARN for every Required-Evidence item, the hypotheses, and the state
of the repository (clean, pushed, public, no big files or checkpoints). Anything MISSING is a mark at risk -- fix it before submitting.

In [ ]:
import platform, subprocess, sys
import matplotlib, numpy, open_clip, pandas, PIL, scipy, sklearn, torch, torchvision, yaml

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
lines = [f"python {sys.version.split()[0]}", f"platform {platform.platform()}", f"gpu {gpu}",
         f"torch {torch.__version__} (cuda {torch.version.cuda})", f"torchvision {torchvision.__version__}", f"open_clip {open_clip.__version__}",
         f"scikit-learn {sklearn.__version__}", f"numpy {numpy.__version__}", f"scipy {scipy.__version__}", f"pandas {pandas.__version__}",
         f"matplotlib {matplotlib.__version__}", f"Pillow {PIL.__version__}", f"PyYAML {yaml.__version__}"]
os.makedirs("env", exist_ok=True)
open("env/colab_environment.txt", "w").write("\n".join(lines) + "\n")
!pip freeze > env/colab_pip_freeze.txt
!python tools/check_submission.py --write-map report/evidence_map.md
!git pull
!git add env report/evidence_map.md tools
!git commit -m "Record the Colab environment; refresh the evidence map"
!git push
!python tools/check_submission.py

## Running a task

Datasets that download automatically (STL-10, CIFAR) live under `DATA_ROOT` on Drive; PACS lives on local
disk at `/content/pacs` (rebuilt each session by the PACS section above). Every config's
`output.checkpoint` should point under `CKPT_ROOT` so a disconnect doesn't lose a checkpoint.

```python
!python -m task2.train --config task2/configs/source_only.yaml --pacs-root /content/pacs
```

After any run that produced new small result files (JSON/CSV/figures, not checkpoints),
commit and push from a cell:

```python
!git add task2/results/*.json report/figures
!git commit -m "Add Task 2 source-only results"
!git push
```